# 🎙️ Обучение персонального офлайн-голоса Piper TTS (Русский язык)

Этот блокнот автоматически:
1. Скачивает ваши аудиозаписи и список фраз из репозитория `Holki9/piper-voice`.
2. Нарезает аудиозаписи с помощью нейросети **Whisper** на 50 отдельных WAV-файлов и формирует `metadata.csv`.
3. Скачивает базовый мужской чекпоинт русского голоса Piper (`ru_RU-dmitri-medium`).
4. Дообучает (fine-tune) модель под ваш персональный тембр (~30–40 минут на бесплатной GPU T4).
5. Экспортирует готовую легковесную модель `my_voice.onnx` (~60 МБ) для телефона или ПК.

---
### 🚀 Как запустить (1 клик):
1. В верхнем меню выберите: **Среда выполнения (Runtime)** -> **Сменить тип среды** -> выберите **T4 GPU**.
2. Нажмите **Среда выполнения (Runtime)** -> **Выполнить всё (Run all)** (или `Ctrl + F9`).
3. Можно свернуть браузер и пить чай — в конце заиграет ваш голос и скачается файл `my_voice.onnx`!

## 1. Проверка GPU и установка зависимостей

In [ ]:
!nvidia-smi

# Установка необходимых системных утилит и библиотек
!apt-get update -qq && apt-get install -y -qq espeak-ng ffmpeg
!pip install -q openai-whisper soundfile librosa onnx
!pip install -q pytorch-lightning==1.9.5 torchmetrics==0.11.4 piper-phonemize

# Клонируем piper для обучения
import os
if not os.path.exists('piper'):
    !git clone --depth 1 https://github.com/rhasspy/piper.git
    !pip install -e ./piper/src/python_run
    !pip install -e ./piper/src/python
print('✓ Зависимости установлены успешно!')

## 2. Скачивание ваших записей и автоматическая нарезка через Whisper

In [ ]:
import json, os, subprocess, whisper, soundfile as sf

# Скачиваем репозиторий с аудиофайлами
if not os.path.exists('piper-voice'):
    !git clone https://github.com/Holki9/piper-voice.git

with open('piper-voice/sentences.json', 'r', encoding='utf-8') as f:
    sentences_data = json.load(f)

os.makedirs('dataset/wavs', exist_ok=True)
metadata = []
sample_counter = 1

print('Загрузка модели Whisper для идеального выравнивания таймингов...')
model = whisper.load_model('base')

for part_key, texts in sentences_data.items():
    audio_path = f'piper-voice/raw_audio/{part_key}.wav'
    print(f'Обработка {audio_path} ({len(texts)} фраз)...')
    
    result = model.transcribe(audio_path, language='ru', word_timestamps=True)
    segments = result['segments']
    
    # Сопоставляем сегменты с фразами
    for i, target_text in enumerate(texts):
        file_id = f'{sample_counter:04d}'
        out_wav = f'dataset/wavs/{file_id}.wav'
        
        if i < len(segments):
            seg = segments[i]
            start_t = max(0, seg['start'] - 0.05)
            dur = seg['end'] - start_t + 0.1
            subprocess.run([
                'ffmpeg', '-y', '-ss', str(start_t), '-t', str(dur),
                '-i', audio_path, '-ar', '22050', '-ac', '1',
                '-af', 'silenceremove=start_periods=1:start_duration=0.02:start_threshold=-40dB,areverse,silenceremove=start_periods=1:start_duration=0.02:start_threshold=-40dB,areverse',
                out_wav
            ], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        metadata.append(f'{file_id}|{target_text}')
        sample_counter += 1

with open('dataset/metadata.csv', 'w', encoding='utf-8') as f:
    f.write('\n'.join(metadata) + '\n')

print(f'✓ Датасет сформирован: {len(metadata)} сэмплов в dataset/metadata.csv')

## 3. Предобработка датасета (фонетизация через espeak-ng)

In [ ]:
# Скачиваем базовую конфигурацию и предобученный чекпоинт
!mkdir -p base_model
!wget -q -O base_model/config.json https://huggingface.co/datasets/rhasspy/piper-checkpoints/resolve/main/ru/ru_RU/dmitri/medium/config.json
!wget -q -O base_model/dmitri.ckpt https://huggingface.co/datasets/rhasspy/piper-checkpoints/resolve/main/ru/ru_RU/dmitri/medium/epoch%3D5589-step%3D1478840.ckpt

# Предобработка
!python3 -m piper_train.preprocess \
    --language ru \
    --input-dir dataset \
    --output-dir training_dir \
    --dataset-format ljspeech \
    --single-speaker \
    --sample-rate 22050

print('✓ Предобработка завершена!')

## 4. Запуск дообучения (Fine-Tuning)

In [ ]:
# Дообучаем модель на GPU T4. 1000–1200 эпох занимают около 25-35 минут
!python3 -m piper_train \
    --dataset-dir training_dir \
    --accelerator gpu \
    --devices 1 \
    --batch-size 16 \
    --validation-split 0.05 \
    --checkpoint-epochs 100 \
    --max_epochs 1200 \
    --resume_from_checkpoint base_model/dmitri.ckpt

## 5. Экспорт в ONNX и тестирование голоса

In [ ]:
import glob, shutil
from IPython.display import Audio, display

# Находим последний сохраненный чекпоинт
checkpoints = sorted(glob.glob('training_dir/lightning_logs/version_*/checkpoints/*.ckpt'), key=os.path.getmtime)
best_ckpt = checkpoints[-1]
print(f'Экспорт чекпоинта: {best_ckpt}')

# Экспортируем в ONNX
!python3 -m piper_train.export_onnx {best_ckpt} my_voice.onnx
shutil.copy('training_dir/config.json', 'my_voice.onnx.json')

# Тестируем синтез
test_phrase = 'Привет! Это мой персональный дикторский голос. Теперь я говорю полностью офлайн прямо на вашем устройстве.'
!echo '{test_phrase}' | piper --model my_voice.onnx --output_file test_output.wav

print('\nСлушайте результат:')
display(Audio('test_output.wav'))

# Скачивание модели в браузер
from google.colab import files
print('Скачивание my_voice.onnx...')
files.download('my_voice.onnx')
files.download('my_voice.onnx.json')